In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('.\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [3]:


def generar_dim_fecha(fecha_inicio: str, fecha_fin: str) -> pd.DataFrame:
    # Genera un rango de todos los días entre las dos fechas
    fechas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')
    dias_es = {
        1: 'Lunes', 2: 'Martes', 3: 'Miércoles', 4: 'Jueves',
        5: 'Viernes', 6: 'Sábado', 7: 'Domingo'
    }
    meses_es = {
        1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
        5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
        9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
    }
    df = pd.DataFrame({'fecha': fechas})
    
    # Clave sustituta: formato YYYYMMDD como entero
    df['key_fecha'] = df['fecha'].dt.strftime('%Y%m%d').astype(int)
    
    # Atributos de negocio
    df['fecha_completa']     = df['fecha'].dt.date
    df['año']                = df['fecha'].dt.year
    df['mes']                = df['fecha'].dt.month
    df['nombre_mes']         = df['fecha'].dt.strftime('%B')  
    df['nombre_mes']         = df['mes'].map(meses_es)      
    df['trimestre']          = df['fecha'].dt.quarter
    df['semana_año']         = df['fecha'].dt.isocalendar().week.astype(int)
    df['dia_mes']            = df['fecha'].dt.day
    df['dia_semana']         = df['fecha'].dt.dayofweek + 1 
       
          
    df['nombre_dia']         = df['fecha'].dt.strftime('%A')  
    df['nombre_dia']         = df['dia_semana'].map(dias_es)       
    df['es_fin_semana']      = df['fecha'].dt.dayofweek >= 5        
    df['es_festivo']         = False 
    
    
    df['es_dia_habil']       = ~df['es_fin_semana'] & ~df['es_festivo']
    
    return df[['key_fecha', 'fecha_completa', 'año', 'trimestre',
               'mes', 'nombre_mes', 'semana_año', 'dia_mes',
               'dia_semana', 'nombre_dia', 'es_fin_semana',
               'es_festivo', 'es_dia_habil']]


dim_fecha = generar_dim_fecha('2020-01-01', '2027-12-31')
dim_fecha.head(10)

,key_fecha,fecha_completa,año,trimestre,mes,nombre_mes,semana_año,dia_mes,dia_semana,nombre_dia,es_fin_semana,es_festivo,es_dia_habil
0,20200101,2020-01-01,2020,1,1,Enero,1,1,3,Miércoles,False,False,True
1,20200102,2020-01-02,2020,1,1,Enero,1,2,4,Jueves,False,False,True
2,20200103,2020-01-03,2020,1,1,Enero,1,3,5,Viernes,False,False,True
3,20200104,2020-01-04,2020,1,1,Enero,1,4,6,Sábado,True,False,False
4,20200105,2020-01-05,2020,1,1,Enero,1,5,7,Domingo,True,False,False
5,20200106,2020-01-06,2020,1,1,Enero,2,6,1,Lunes,False,False,True
6,20200107,2020-01-07,2020,1,1,Enero,2,7,2,Martes,False,False,True
7,20200108,2020-01-08,2020,1,1,Enero,2,8,3,Miércoles,False,False,True
8,20200109,2020-01-09,2020,1,1,Enero,2,9,4,Jueves,False,False,True
9,20200110,2020-01-10,2020,1,1,Enero,2,10,5,Viernes,False,False,True


In [4]:
dim_fecha.to_sql('dim_fecha', etl_conn, if_exists='replace', index=False)

922